# 3a — Modèles sensibles à l'échelle (CRISP-DM Phase 4)

Cette famille regroupe les modèles dont la fonction objectif ou la métrique de similarité dépend de l'**échelle** des features — donc **tous** doivent recevoir des données normalisées (`preprocessor_scaled`).

| Modèle           | Hypothèse principale                                                                                  |
|------------------|--------------------------------------------------------------------------------------------------------|
| OLS              | Linéarité, résidus normaux et homoscédastiques, pas de multicolinéarité parfaite                       |
| Ridge            | Idem OLS, tolère la multicolinéarité par shrinkage — features comparables en échelle indispensables   |
| Lasso            | Idem Ridge + parcimonie : suppose qu'une minorité de coefficients sont non-nuls                        |
| ElasticNet       | Compromis Ridge/Lasso                                                                                  |
| KNN              | Distance euclidienne pertinente — features comparables en échelle indispensables                       |
| MLPRegressor     | Approximation lisse par réseau de neurones — descente de gradient sensible à l'échelle                |

In [ ]:
%load_ext autoreload
%autoreload 2
%run 2_data_prep.ipynb

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, Lasso, LassoCV, ElasticNet, ElasticNetCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

FAMILY = 'scaled'

## 4.1 OLS (régression linéaire ordinaire) — baseline

### Hypothèses du modèle
1. **Linéarité** entre les features et `y_log` — vérifiable par les résidus-vs-fitted.
2. **Indépendance** des erreurs.
3. **Homoscédasticité** (variance constante des résidus) — vérifiable par scale-location.
4. **Normalité** des résidus — vérifiable par Q-Q plot.
5. **Pas de multicolinéarité parfaite** — partiellement adressée par T05 (drops `GarageArea`/`TotalBsmtSF`/`TotRmsAbvGrd`/`GarageYrBlt`, à venir).

**Préprocesseur** : `preprocessor_scaled` — OLS n'a pas strictement besoin de la mise à l'échelle (les coefficients absorbent l'unité), mais on l'inclut pour cohérence et meilleure stabilité numérique.

In [ ]:
t0 = time.time()
ols_pipe = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('model', LinearRegression()),
])
ols_pipe.fit(X_train, y_train_log)
ols_fit_s = time.time() - t0

ols_pred_log = ols_pipe.predict(X_test)
ols_holdout = rmsle_score(y_test_log, ols_pred_log)
ols_cv = cv_rmsle(ols_pipe, X_train, y_train_log)

print(f"OLS — CV RMSLE: {ols_cv:.4f}   Holdout RMSLE: {ols_holdout:.4f}   fit_time: {ols_fit_s:.2f}s")
predicted_vs_actual_plot(y_test_log, ols_pred_log, title=f"OLS — Holdout RMSLE: {ols_holdout:.4f}")
plt.show()

publish_result(FAMILY, 'OLS', cv_rmsle=ols_cv, holdout_rmsle=ols_holdout, fit_time_s=ols_fit_s)

## 4.2 Ridge — pénalité L2

### Hypothèses du modèle
- Mêmes que OLS, mais Ridge **shrinke** les coefficients vers 0 sans en annuler aucun. Tolère la multicolinéarité (un peu) et stabilise l'estimation.
- L'hypothèse cruciale : les **features sont comparables en échelle**, sinon la pénalité L2 (somme des carrés des coefficients) frappe les features de grande amplitude davantage que les autres. → `preprocessor_scaled` est **mandatory**.

On utilise `RidgeCV` pour sélectionner automatiquement le meilleur `alpha` par cross-validation interne.

In [ ]:
t0 = time.time()
ridge_pipe = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('model', RidgeCV(alphas=np.logspace(-2, 2, 20), cv=5)),
])
ridge_pipe.fit(X_train, y_train_log)
ridge_fit_s = time.time() - t0
ridge_alpha = ridge_pipe.named_steps['model'].alpha_

ridge_pred_log = ridge_pipe.predict(X_test)
ridge_holdout = rmsle_score(y_test_log, ridge_pred_log)
ridge_cv = cv_rmsle(ridge_pipe, X_train, y_train_log)

print(f"Ridge — CV RMSLE: {ridge_cv:.4f}   Holdout RMSLE: {ridge_holdout:.4f}   alpha*: {ridge_alpha:.4g}   fit_time: {ridge_fit_s:.2f}s")
predicted_vs_actual_plot(y_test_log, ridge_pred_log, title=f"Ridge (alpha={ridge_alpha:.3g}) — RMSLE: {ridge_holdout:.4f}")
plt.show()

publish_result(FAMILY, 'Ridge', cv_rmsle=ridge_cv, holdout_rmsle=ridge_holdout, fit_time_s=ridge_fit_s, params={'alpha': ridge_alpha})

## 4.3 Lasso — pénalité L1

### Hypothèses du modèle
- Idem Ridge, mais avec pénalité L1 (somme des **valeurs absolues**) — provoque la **sélection de variables** : certains coefficients sont mis à exactement 0.
- Hypothèse implicite : le « vrai » modèle est **parcimonieux** (la plupart des features ne contribuent pas).
- L'échelle est encore plus critique qu'avec Ridge.

In [ ]:
t0 = time.time()
lasso_pipe = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('model', LassoCV(alphas=np.logspace(-4, 2, 20), cv=5, max_iter=50_000, random_state=RANDOM_STATE)),
])
lasso_pipe.fit(X_train, y_train_log)
lasso_fit_s = time.time() - t0
lasso_alpha = lasso_pipe.named_steps['model'].alpha_

lasso_pred_log = lasso_pipe.predict(X_test)
lasso_holdout = rmsle_score(y_test_log, lasso_pred_log)
lasso_cv = cv_rmsle(lasso_pipe, X_train, y_train_log)

coefs = lasso_pipe.named_steps['model'].coef_
n_zero = int((coefs == 0).sum())
print(f"Lasso — CV RMSLE: {lasso_cv:.4f}   Holdout RMSLE: {lasso_holdout:.4f}   alpha*: {lasso_alpha:.4g}   fit_time: {lasso_fit_s:.2f}s")
print(f"  variables annulées : {n_zero} / {len(coefs)} ({100*n_zero/len(coefs):.0f}%)")
predicted_vs_actual_plot(y_test_log, lasso_pred_log, title=f"Lasso (alpha={lasso_alpha:.3g}) — RMSLE: {lasso_holdout:.4f}")
plt.show()

publish_result(FAMILY, 'Lasso', cv_rmsle=lasso_cv, holdout_rmsle=lasso_holdout, fit_time_s=lasso_fit_s,
               params={'alpha': lasso_alpha, 'n_zero_coefs': n_zero, 'n_total_coefs': len(coefs)})

## 4.4 ElasticNet — compromis L1 + L2

### Hypothèses du modèle
- Mélange Ridge et Lasso via le paramètre `l1_ratio`. Sélectionne des features (comme Lasso) tout en partageant la pénalité entre plusieurs features corrélées (comme Ridge).
- Pertinent quand on a des **groupes de features corrélées** — exactement notre cas (cf. NB1 : `GarageCars`/`GarageArea`, `TotalBsmtSF`/`1stFlrSF`).

In [ ]:
t0 = time.time()
enet_pipe = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('model', ElasticNetCV(
        l1_ratio=[.1, .5, .7, .9, .95, .99, 1],
        alphas=np.logspace(-4, 2, 20),
        cv=5, max_iter=50_000, random_state=RANDOM_STATE,
    )),
])
enet_pipe.fit(X_train, y_train_log)
enet_fit_s = time.time() - t0
enet_alpha = enet_pipe.named_steps['model'].alpha_
enet_l1 = enet_pipe.named_steps['model'].l1_ratio_

enet_pred_log = enet_pipe.predict(X_test)
enet_holdout = rmsle_score(y_test_log, enet_pred_log)
enet_cv = cv_rmsle(enet_pipe, X_train, y_train_log)

print(f"ElasticNet — CV RMSLE: {enet_cv:.4f}   Holdout RMSLE: {enet_holdout:.4f}   alpha*: {enet_alpha:.4g}   l1_ratio*: {enet_l1}   fit_time: {enet_fit_s:.2f}s")
predicted_vs_actual_plot(y_test_log, enet_pred_log, title=f"ElasticNet — RMSLE: {enet_holdout:.4f}")
plt.show()

publish_result(FAMILY, 'ElasticNet', cv_rmsle=enet_cv, holdout_rmsle=enet_holdout, fit_time_s=enet_fit_s,
               params={'alpha': enet_alpha, 'l1_ratio': enet_l1})

## 4.5 KNN — k plus proches voisins

### Hypothèses du modèle
- **Aucune hypothèse paramétrique** sur la forme de la relation features↔cible — non-paramétrique.
- Hypothèse implicite : deux points proches dans l'espace des features ont des valeurs `y` proches → la **distance euclidienne** doit avoir un sens.
- Conséquence directe : **toutes les features doivent être à la même échelle**, sans quoi les features de grande amplitude dominent la distance. → `preprocessor_scaled` indispensable.

On balaie `k ∈ [1, 40]` et on choisit le `k` qui maximise le R² en CV.

In [ ]:
t0 = time.time()
k_range = list(range(1, 41))
knn_cv_scores = []
for k in k_range:
    pipe = Pipeline([('preprocessor', preprocessor_scaled), ('model', KNeighborsRegressor(n_neighbors=k))])
    knn_cv_scores.append(-cross_val_score(pipe, X_train, y_train_log, cv=5,
                                          scoring='neg_root_mean_squared_error', n_jobs=-1).mean())

best_k = int(k_range[int(np.argmin(knn_cv_scores))])

plt.figure(figsize=(9, 5))
plt.plot(k_range, knn_cv_scores, marker='o')
plt.axvline(best_k, color='red', ls='--', alpha=0.6, label=f'k* = {best_k}')
plt.xlabel('k (nombre de voisins)')
plt.ylabel('CV RMSLE (5-fold)')
plt.title("KNN — balayage de k")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

knn_pipe = Pipeline([('preprocessor', preprocessor_scaled), ('model', KNeighborsRegressor(n_neighbors=best_k))])
knn_pipe.fit(X_train, y_train_log)
knn_fit_s = time.time() - t0
knn_pred_log = knn_pipe.predict(X_test)
knn_holdout = rmsle_score(y_test_log, knn_pred_log)
knn_cv = float(min(knn_cv_scores))

print(f"KNN — CV RMSLE: {knn_cv:.4f}   Holdout RMSLE: {knn_holdout:.4f}   k*: {best_k}   fit_time (sweep + final): {knn_fit_s:.2f}s")
predicted_vs_actual_plot(y_test_log, knn_pred_log, title=f"KNN (k={best_k}) — RMSLE: {knn_holdout:.4f}")
plt.show()

publish_result(FAMILY, 'KNN', cv_rmsle=knn_cv, holdout_rmsle=knn_holdout, fit_time_s=knn_fit_s, params={'k': best_k})

## 4.6 MLPRegressor — réseau de neurones tabulaire

### Hypothèses du modèle
- Approximation de fonction lisse par couches denses + activation non-linéaire.
- **Descente de gradient** : très sensible à l'échelle des features (sans normalisation, le gradient explose dans certaines directions et stagne dans d'autres).
- Nécessite une **taille de couche cachée** raisonnable et un **early-stopping** pour éviter le surapprentissage.

On utilise une architecture modeste (1 couche de 64 unités) car notre jeu est petit (≈ 1200 obs train).

In [ ]:
t0 = time.time()
mlp_pipe = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('model', MLPRegressor(
        hidden_layer_sizes=(64,),
        activation='relu',
        solver='adam',
        max_iter=1500,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=RANDOM_STATE,
    )),
])
mlp_pipe.fit(X_train, y_train_log)
mlp_fit_s = time.time() - t0
mlp_pred_log = mlp_pipe.predict(X_test)
mlp_holdout = rmsle_score(y_test_log, mlp_pred_log)
mlp_cv = cv_rmsle(mlp_pipe, X_train, y_train_log)

print(f"MLP — CV RMSLE: {mlp_cv:.4f}   Holdout RMSLE: {mlp_holdout:.4f}   fit_time: {mlp_fit_s:.2f}s")
predicted_vs_actual_plot(y_test_log, mlp_pred_log, title=f"MLPRegressor — RMSLE: {mlp_holdout:.4f}")
plt.show()

publish_result(FAMILY, 'MLPRegressor', cv_rmsle=mlp_cv, holdout_rmsle=mlp_holdout, fit_time_s=mlp_fit_s,
               params={'hidden_layer_sizes': (64,), 'activation': 'relu', 'solver': 'adam'})

## 4.7 Comparaison intra-famille

On compare les RMSLE des six modèles sensibles à l'échelle. C'est cette comparaison qui désigne le **champion de famille** qui sera proposé comme base learner dans le notebook `3d_stacking.ipynb`.

In [ ]:
family_path = RESULTS_DIR / f'family_{FAMILY}.json'
fam = pd.DataFrame(json.loads(family_path.read_text()))
fam = fam.sort_values('holdout_rmsle').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(fam['model'], fam['holdout_rmsle'], color=sns.color_palette('viridis', len(fam)))
for bar, v in zip(bars, fam['holdout_rmsle']):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.002, f"{v:.4f}", ha='center', fontweight='bold')
ax.set_ylabel('RMSLE (holdout)')
ax.set_title("Famille « scaled » — RMSLE par modèle (plus bas = mieux)")
ax.set_ylim(0, fam['holdout_rmsle'].max() * 1.15)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print("Classement intra-famille (par holdout RMSLE) :")
display(fam[['model', 'cv_rmsle', 'holdout_rmsle', 'fit_time_s']])